In [1]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm import tqdm


In [2]:
from JAX_BSSN.evolve import rk4_step
from JAX_BSSN.bssn import BSSNVariables, BSSNParameters, compute_momentum_constraint
from JAX_BSSN.derivatives import diff1_field
from JAX_BSSN.errors import compute_hamiltonian_constraint
from JAX_BSSN.tensor_algebra import (
    invert_3x3_metric,
    christoffel_symbols_second_kind,
    trace_tensor,
    traceless_part,
)


This notebook implements the **1D linear wave test** from section 4.3 of
*Alcubierre et al., CQG 21 (2004) 589*.

Test metric (Gauss coordinates):

$$
ds^2 = -dt^2 + dx^2 + (1+b)\,dy^2 + (1-b)\,dz^2,
$$

with

$$
b(x,t) = A\sin\left(\frac{2\pi(x-t)}{d}\right).
$$

Non-trivial extrinsic curvature components:

$$
K_{yy} = -\tfrac{1}{2}\partial_t b, \qquad K_{zz} = +\tfrac{1}{2}\partial_t b.
$$

(Using the same sign convention as the code: $K_{ij} = -\tfrac{1}{2}\partial_t g_{ij}$ for $\alpha=1,\beta^i=0$.)

Like the gauge-wave notebook, this one includes:
- initialization and evolution,
- norms and profile plots,
- analytic error tracking,
- self-convergence diagnostics.


Paper settings (for reference):

- Amplitude: `A = 1e-8`
- Domain: `x in [-0.5, 0.5]`, `d = 1`
- Resolutions: `rho = 1, 2, 4` -> `nx = 50*rho`
- Time step: `dt = dx/4`
- Runtime: `T = 1000` crossing times
- Output cadence: every `10` crossing times

The defaults below are shorter for interactive runs. Increase `T_crossings` and
`save_every_crossings` to match the full paper run.


In [3]:
# Simulation controls
rhos = [1, 2, 4]
nxs = [50 * rho for rho in rhos]

L = 1.0
A = 1.0e-8
d = 1.0

# Interactive defaults (paper uses T_crossings=1000, save_every_crossings=10)
T_crossings = 100
save_every_crossings = 1

print("nxs:", nxs)
print("A:", A, "d:", d)
print("T_crossings:", T_crossings)
print("save_every_crossings:", save_every_crossings)


nxs: [50, 100, 200]
A: 1e-08 d: 1.0
T_crossings: 100
save_every_crossings: 1


In [4]:
def make_cell_centered_grid(nx, L=1.0):
    dx = L / nx
    x = -L / 2 + (jnp.arange(nx) + 0.5) * dx
    return x, dx


def linear_wave_b(x, t, A, d):
    return A * jnp.sin((2.0 * jnp.pi * (x - t)) / d)


def l2_norm_over_x(history, dx):
    return jnp.sqrt(dx * jnp.sum(history**2, axis=1))


def linf_norm_over_x(history):
    return jnp.max(jnp.abs(history), axis=1)


def restrict_history_cell_centered(history, refinement_ratio):
    """
    Restrict cell-centered 1D data by repeated pair-averaging.
    """
    out = history
    ratio = int(refinement_ratio)
    while ratio > 1:
        out = 0.5 * (out[:, 0::2] + out[:, 1::2])
        ratio //= 2
    return out


In [5]:
def initialize_linear_wave_state(x, dx, dt, A, d):
    nx = x.shape[0]

    b0 = linear_wave_b(x, 0.0, A, d)
    g_xx = jnp.ones_like(x)
    g_yy = 1.0 + b0
    g_zz = 1.0 - b0

    # W = det(g_ij)^(-1/6), with det(g_ij) = (1+b)(1-b) = 1-b^2
    W = jnp.power(1.0 - b0**2, -1.0 / 6.0)

    gamma = jnp.zeros((3, 3, nx, 1, 1))
    gamma = gamma.at[0, 0, :, 0, 0].set(g_xx * W**2)
    gamma = gamma.at[1, 1, :, 0, 0].set(g_yy * W**2)
    gamma = gamma.at[2, 2, :, 0, 0].set(g_zz * W**2)

    dbdt0 = -(2.0 * jnp.pi * A / d) * jnp.cos((2.0 * jnp.pi * x) / d)

    K = jnp.zeros_like(gamma)
    K = K.at[1, 1, :, 0, 0].set(-0.5 * dbdt0)
    K = K.at[2, 2, :, 0, 0].set(0.5 * dbdt0)

    inv_gamma = invert_3x3_metric(gamma)
    A_ij = W**2 * traceless_part(K, gamma, inv_gamma)
    K_trace = jnp.expand_dims(W, axis=(1, 2))**2 * trace_tensor(K, inv_gamma)

    derivs = jnp.stack([diff1_field(gamma, d_ + 2, dx) for d_ in range(3)], axis=0)
    christoffel_2 = christoffel_symbols_second_kind(inv_gamma, derivs)
    conformal_connection = jnp.einsum('mn...,imn...->i...', inv_gamma, christoffel_2)

    lapse = jnp.ones((nx, 1, 1))
    shift = jnp.zeros((3, nx, 1, 1))

    vars0 = BSSNVariables(
        conformal_metric=gamma,
        conformal_factor=jnp.expand_dims(W, axis=(1, 2)),
        traceless_K=A_ij,
        trace_K=K_trace,
        conformal_connection=conformal_connection,
        lapse=lapse,
        shift=shift,
    )

    params = BSSNParameters(
        eta=0.0,
        kappa=0.0,
        nu=0.0,
        f=0.0,  # freeze lapse for Gauss-coordinate test
        g=0.0,
        dx=dx,
        dt=dt,
    )

    return vars0, params


def run_linear_wave(nx, A, d, L=1.0, T_crossings=1.0, save_every_crossings=0.1, show_progress=True):
    x, dx = make_cell_centered_grid(nx, L=L)
    dt = 0.25 * dx

    steps_per_crossing = int(round(d / dt))
    Nt = int(round(T_crossings * steps_per_crossing))
    snapshot_stride = max(1, int(round(save_every_crossings * steps_per_crossing)))

    vars, params = initialize_linear_wave_state(x, dx, dt, A, d)

    times = [0.0]
    pm = vars.conformal_metric / (vars.conformal_factor**2)
    gyy_hist = [pm[1, 1, :, 0, 0]]
    gzz_hist = [pm[2, 2, :, 0, 0]]
    alpha_hist = [vars.lapse[:, 0, 0]]
    ham_hist = [compute_hamiltonian_constraint(vars, params)]
    mom_hist = [compute_momentum_constraint(vars, params)]

    iterator = range(1, Nt + 1)
    if show_progress:
        iterator = tqdm(iterator, leave=False)

    for step in iterator:
        vars = rk4_step(vars, params)
        if (step % snapshot_stride == 0) or (step == Nt):
            t = step * dt
            pm = vars.conformal_metric / (vars.conformal_factor**2)
            times.append(t)
            gyy_hist.append(pm[1, 1, :, 0, 0])
            gzz_hist.append(pm[2, 2, :, 0, 0])
            alpha_hist.append(vars.lapse[:, 0, 0])
            ham_hist.append(compute_hamiltonian_constraint(vars, params))
            mom_hist.append(compute_momentum_constraint(vars, params))

    times = jnp.asarray(times)
    gyy_hist = jnp.asarray(gyy_hist)
    gzz_hist = jnp.asarray(gzz_hist)
    alpha_hist = jnp.asarray(alpha_hist)

    b_exact = jax.vmap(lambda t: linear_wave_b(x, t, A, d))(times)
    gyy_exact = 1.0 + b_exact
    gzz_exact = 1.0 - b_exact
    alpha_exact = jnp.ones_like(gzz_exact)

    return {
        "nx": nx,
        "x": x,
        "dx": dx,
        "dt": dt,
        "Nt": Nt,
        "snapshot_stride": snapshot_stride,
        "times": times,
        "gyy": gyy_hist,
        "gzz": gzz_hist,
        "alpha": alpha_hist,
        "gyy_exact": gyy_exact,
        "gzz_exact": gzz_exact,
        "alpha_exact": alpha_exact,
        "ham": jnp.asarray(ham_hist),
        "mom": jnp.asarray(mom_hist),
    }


In [ ]:
results = {}
for nx in nxs:
    print(f"Running nx={nx} ...")
    results[nx] = run_linear_wave(
        nx=nx,
        A=A,
        d=d,
        L=L,
        T_crossings=T_crossings,
        save_every_crossings=save_every_crossings,
        show_progress=True,
    )

# Check time alignment across resolutions
t_ref = results[nxs[0]]["times"]
for nx in nxs[1:]:
    t_n = results[nx]["times"]
    if t_n.shape != t_ref.shape:
        raise ValueError(f"Snapshot-count mismatch for nx={nx}: {t_n.shape} vs {t_ref.shape}")
    print(f"max |dt_align| for nx={nx}:", float(jnp.max(jnp.abs(t_n - t_ref))))

print("Done.")


Running nx=50 ...


Running nx=100 ...


Running nx=200 ...


 83%|████████▎ | 66436/80000 [14:24:28<1:21:38,  2.77it/s]     

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for nx in nxs:
    r = results[nx]
    t = r["times"]

    gzz_l2 = l2_norm_over_x(r["gzz"], r["dx"])
    alpha_max = jnp.max(jnp.abs(r["alpha"]), axis=1)

    axes[0].plot(t, gzz_l2, marker=".", linewidth=1, label=f"{nx} points")
    axes[1].plot(t, alpha_max, marker=".", linewidth=1, label=f"{nx} points")

axes[0].set_xlabel("Time (crossing times)")
axes[0].set_ylabel(r"$||g_{zz}||_2$")
axes[0].set_title(r"$L^2$ norm of $g_{zz}$")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Time (crossing times)")
axes[1].set_ylabel(r"$\max(|\alpha|)$")
axes[1].set_title("Max lapse")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Figure-4 style profile of gzz - 1

# Use mid-run profile when possible (paper shows profile at 500 crossings for a 1000-crossing run)
profile_time = float(min(0.5 * T_crossings, results[nxs[0]]["times"][-1]))
idx = int(jnp.argmin(jnp.abs(results[nxs[0]]["times"] - profile_time)))
t_profile = float(results[nxs[0]]["times"][idx])

plt.figure(figsize=(8, 4))

# Smooth exact curve
x_exact = jnp.linspace(-0.5, 0.5, 2000, endpoint=False)
gzz_exact_curve = 1.0 - linear_wave_b(x_exact, t_profile, A, d)
plt.plot(x_exact, gzz_exact_curve - 1.0, "k--", linewidth=1.5, label="Exact solution")

for nx in nxs:
    r = results[nx]
    plt.plot(r["x"], r["gzz"][idx] - 1.0, marker=".", linewidth=1, label=f"{nx} points")

plt.xlabel("x")
plt.ylabel(r"$g_{zz} - 1$")
plt.title(f"Linear wave profile at t={t_profile:.3f} crossings")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

final_linf_errors = []

for nx in nxs:
    r = results[nx]
    t = r["times"]

    err_gzz = r["gzz"] - r["gzz_exact"]
    linf_err = linf_norm_over_x(err_gzz)
    l2_err = l2_norm_over_x(err_gzz, r["dx"])

    final_linf_errors.append(linf_err[-1])

    axes[0].plot(t, linf_err, marker=".", linewidth=1, label=f"{nx} points")
    axes[1].plot(t, l2_err, marker=".", linewidth=1, label=f"{nx} points")

axes[0].set_yscale("log")
axes[1].set_yscale("log")

axes[0].set_xlabel("Time (crossing times)")
axes[0].set_ylabel(r"$||g_{zz}^{num} - g_{zz}^{exact}||_{\infty}$")
axes[0].set_title(r"$L^{\infty}$ error of $g_{zz}$")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Time (crossing times)")
axes[1].set_ylabel(r"$||g_{zz}^{num} - g_{zz}^{exact}||_2$")
axes[1].set_title(r"$L^2$ error of $g_{zz}$")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Pairwise observed order from final-time L_inf errors
p_50_100 = float(jnp.log2(final_linf_errors[0] / final_linf_errors[1]))
p_100_200 = float(jnp.log2(final_linf_errors[1] / final_linf_errors[2]))
print(f"Final-time L_inf order gzz: p(50->100)={p_50_100:.3f}, p(100->200)={p_100_200:.3f}")


In [ ]:
def self_convergence_factor(coarse_hist, mid_hist, fine_hist, dx_coarse, ratio_mid, ratio_fine, den_floor=1e-30):
    mid_on_coarse = restrict_history_cell_centered(mid_hist, ratio_mid)
    fine_on_coarse = restrict_history_cell_centered(fine_hist, ratio_fine)

    diff_c_m = l2_norm_over_x(coarse_hist - mid_on_coarse, dx_coarse)
    diff_m_f = l2_norm_over_x(mid_on_coarse - fine_on_coarse, dx_coarse)

    valid = diff_m_f > den_floor
    cf = jnp.where(valid, diff_c_m / diff_m_f, jnp.nan)
    p = jnp.where(valid, jnp.log2(cf), jnp.nan)
    return cf, p, valid


nx_c, nx_m, nx_f = nxs
ratio_m = nx_m // nx_c
ratio_f = nx_f // nx_c

t = results[nx_c]["times"]
dx_c = results[nx_c]["dx"]

cf_gzz, p_gzz, valid_gzz = self_convergence_factor(
    results[nx_c]["gzz"], results[nx_m]["gzz"], results[nx_f]["gzz"], dx_c, ratio_m, ratio_f
)

cf_alpha, p_alpha, valid_alpha = self_convergence_factor(
    results[nx_c]["alpha"], results[nx_m]["alpha"], results[nx_f]["alpha"], dx_c, ratio_m, ratio_f
)

# Report early-time order where CF is numerically well-defined.
initial_slice = slice(1, min(6, p_gzz.shape[0]))

def nanmean_safe(arr):
    m = jnp.isfinite(arr)
    return float(jnp.sum(jnp.where(m, arr, 0.0)) / jnp.maximum(jnp.sum(m), 1)), int(jnp.sum(m))

initial_order_gzz, n_gzz = nanmean_safe(p_gzz[initial_slice])
initial_order_alpha, n_alpha = nanmean_safe(p_alpha[initial_slice])

if n_gzz > 0:
    print(f"Initial self-convergence order (gzz):   {initial_order_gzz:.3f}")
else:
    print("Initial self-convergence order (gzz):   undefined (no valid CF samples)")

if n_alpha > 0:
    print(f"Initial self-convergence order (lapse): {initial_order_alpha:.3f}")
else:
    print("Initial self-convergence order (lapse): undefined (lapse is nearly constant; roundoff-dominated)")

print("Reference for exact 2nd order: CF=4 and p=2")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].plot(t, cf_gzz, marker=".", linewidth=1, label=r"CF($g_{zz}$)")
axes[0].plot(t, cf_alpha, marker=".", linewidth=1, label=r"CF($\alpha$)")
axes[0].axhline(4.0, color="k", linestyle="--", linewidth=1, label="Exact 2nd order")
axes[0].set_xlabel("Time (crossing times)")
axes[0].set_ylabel("Convergence factor")
axes[0].set_title("Self-convergence factors")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(t, p_gzz, marker=".", linewidth=1, label=r"p($g_{zz}$)")
axes[1].plot(t, p_alpha, marker=".", linewidth=1, label=r"p($\alpha$)")
axes[1].axhline(2.0, color="k", linestyle="--", linewidth=1, label="Exact 2nd order")
axes[1].set_xlabel("Time (crossing times)")
axes[1].set_ylabel("Observed order")
axes[1].set_title(r"$p=\log_2(\mathrm{CF})$")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Constraint diagnostics
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for nx in nxs:
    r = results[nx]
    t = r["times"]

    ham = r["ham"].reshape((r["ham"].shape[0], -1))
    mom = r["mom"].reshape((r["mom"].shape[0], -1))

    ham_l2 = jnp.sqrt(jnp.mean(ham**2, axis=1))
    mom_l2 = jnp.sqrt(jnp.mean(mom**2, axis=1))

    axes[0].plot(t, ham_l2, marker=".", linewidth=1, label=f"{nx} points")
    axes[1].plot(t, mom_l2, marker=".", linewidth=1, label=f"{nx} points")

axes[0].set_yscale("log")
axes[1].set_yscale("log")

axes[0].set_xlabel("Time (crossing times)")
axes[0].set_ylabel(r"$||\mathcal{H}||_2$")
axes[0].set_title("Hamiltonian constraint")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Time (crossing times)")
axes[1].set_ylabel(r"$||\mathcal{M}||_2$")
axes[1].set_title("Momentum constraint")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
print("Suggested paper-scale settings:")
print("  T_crossings = 1000")
print("  save_every_crossings = 10")
print("Current settings:")
print("  T_crossings =", T_crossings)
print("  save_every_crossings =", save_every_crossings)
for nx in nxs:
    r = results[nx]
    print(f"nx={nx}: dt={float(r['dt']):.6e}, Nt={r['Nt']}, snapshot_stride={r['snapshot_stride']}")

print("\nNote: in this linear-wave test (Gauss coordinates), lapse is analytically constant (alpha=1).")
print("Self-convergence of alpha can therefore become roundoff-dominated.")
